# Notebook 2. Cấp 3, cấp 4, RQ1 và các ablation

Người phụ trách: Phạm Thiên Phúc (cấp 3, RQ1, bộ thế cờ), cấp 4: Nguyễn Trọng Phúc.

Nội dung:
1. Bộ thế cờ chiến thuật `files/tactics.json` và độ chính xác của cấp 1–4.
2. Ablation 1 (bắt buộc): Alpha-Beta bật/tắt cắt tỉa, kèm thứ tự xét nước.
3. Ablation 2 (bắt buộc): số mô phỏng của MCTS.
4. Phân tích độ nhạy hàm lượng giá của cấp 3.
5. RQ1: cấp 3 và cấp 4 với cùng thời gian mỗi nước (0,2 / 0,5 / 1,0 giây).

**Lưu ý khi chạy RQ1:** chế độ thời gian đo theo đồng hồ thật, nên lúc chạy mục 5 không chạy thêm việc nặng nào khác trên máy (không chạy song song notebook khác).

In [ ]:
# QUICK = True: chạy thử vài ván, vài epoch để kiểm tra code (ghi vào thư mục con thu_nhanh/)
# QUICK = False: chạy thật theo config.yaml, kết quả dùng cho báo cáo
QUICK = False

import os
if not os.path.exists("config.yaml"):
    raise RuntimeError("Hãy mở Jupyter trong thư mục final_connect_four (nơi có config.yaml).")

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.experiments import setup, summaries_table, with_random_opening, move_times_of
from utils.arena import run_match, format_summary, read_log, first_player_stats
from utils.data import machine_info
from utils.conn_simple_env import conn
from utils import alphabeta, tactics, rules
from utils.experiments import plot_scores, info_mean, records_of
from utils.arena import module_info
ex = setup(QUICK)
cfg = ex.cfg
print("Cấu hình máy:", machine_info())

## 1. Bộ thế cờ chiến thuật

Ba nhóm thế cờ: thắng ngay, buộc chặn, tạo đòn đôi. Đáp án lấy bằng Alpha-Beta tìm sâu 5 nước (xem `utils/tactics.py`).
Bộ thế cờ được dùng lại ở nb3, nb4, nb6 nên phải chạy mục này trước.

In [ ]:
tactics_path = ex.model("tactics.json")
if os.path.exists(tactics_path):
    positions = tactics.load_tactics(tactics_path)
    print("Đã có bộ thế cờ, nạp lại từ", tactics_path)
else:
    start = time.perf_counter()
    if QUICK:
        positions = tactics.build_tactics(per_group={"win_now": 3, "must_block": 3, "double_threat": 3}, depth=3)
    else:
        positions = tactics.build_tactics()
    tactics.save_tactics(positions, tactics_path)
    print("Sinh bộ thế cờ mất", round(time.perf_counter() - start, 1), "giây")
print("Số thế mỗi nhóm:", tactics.describe(positions))

In [ ]:
rows = []
for name in ["cap1", "cap2", "cap3", "cap4"]:
    move_fn, info_fn = ex.agent(name)
    acc = tactics.score_agent(move_fn, positions)
    row = {"tác tử": name}
    row.update(acc)
    rows.append(row)
tactics_table = pd.DataFrame(rows).round(3)
tactics_table.to_csv(ex.table("nb2_chien_thuat_cap1_4.csv"), index=False, encoding="utf-8-sig")
tactics_table

## 2. Ablation 1: bật/tắt cắt tỉa Alpha-Beta

Trên cùng một bộ thế cờ (lấy từ các ván ngẫu nhiên có seed), so ba cấu hình:
- không cắt tỉa;
- cắt tỉa, xét nước theo thứ tự 1 → 7;
- cắt tỉa, xét nước từ giữa ra (cấu hình thật của cấp 3).

Đo số nút và thời gian, đồng thời kiểm tra điểm các cột ở gốc có giống hệt nhau không (cắt tỉa chỉ được bỏ phép tính thừa, không được đổi kết quả).

In [ ]:
import random

def sample_positions(n, seed):
    """n thế cờ giữa ván từ các ván ngẫu nhiên có seed (dừng ở nước thứ 6 đến 20)."""
    rng = random.Random(seed)
    result = []
    while len(result) < n:
        env = conn()
        stop_at = rng.randint(6, 20)
        while not env.done and len(env.history) < stop_at:
            env.step(rng.choice(env.validinputs))
        if not env.done:
            result.append(env)
    return result

prune_cfg = cfg["ablation_pruning"]
prune_positions = sample_positions(ex.n(prune_cfg["positions"], 5), prune_cfg["seed"])
depths = prune_cfg["depths"] if not QUICK else [2, 3]

def run_config(env, depth, prune, order):
    alphabeta.MOVE_ORDER = order
    search = alphabeta.Search(prune=prune)
    start = time.perf_counter()
    scores = search.root_scores(env, depth)
    return scores, search.nodes, time.perf_counter() - start

CENTER_FIRST = [4, 3, 5, 2, 6, 1, 7]
LEFT_TO_RIGHT = [1, 2, 3, 4, 5, 6, 7]

rows = []
for depth in depths:
    nodes = {"khong_cat": [], "cat_1_7": [], "cat_giua": []}
    seconds = {"khong_cat": [], "cat_1_7": [], "cat_giua": []}
    same = 0
    for env in prune_positions:
        s0, n0, t0 = run_config(env, depth, False, CENTER_FIRST)
        s1, n1, t1 = run_config(env, depth, True, LEFT_TO_RIGHT)
        s2, n2, t2 = run_config(env, depth, True, CENTER_FIRST)
        nodes["khong_cat"].append(n0)
        nodes["cat_1_7"].append(n1)
        nodes["cat_giua"].append(n2)
        seconds["khong_cat"].append(t0)
        seconds["cat_1_7"].append(t1)
        seconds["cat_giua"].append(t2)
        if s0 == s1 == s2:
            same += 1
    rows.append({
        "độ sâu": depth,
        "nút: không cắt": round(np.mean(nodes["khong_cat"])),
        "nút: cắt, 1→7": round(np.mean(nodes["cat_1_7"])),
        "nút: cắt, giữa trước": round(np.mean(nodes["cat_giua"])),
        "ms: không cắt": round(np.mean(seconds["khong_cat"]) * 1000, 1),
        "ms: cắt, 1→7": round(np.mean(seconds["cat_1_7"]) * 1000, 1),
        "ms: cắt, giữa trước": round(np.mean(seconds["cat_giua"]) * 1000, 1),
        "điểm gốc giống nhau": f"{same}/{len(prune_positions)}",
    })
    print("xong độ sâu", depth)

# trả lại thứ tự thật cho các mục sau
alphabeta.MOVE_ORDER = CENTER_FIRST
prune_table = pd.DataFrame(rows)
prune_table.to_csv(ex.table("nb2_ablation_cat_tia.csv"), index=False, encoding="utf-8-sig")
prune_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(prune_table["độ sâu"], prune_table["nút: không cắt"], marker="o", label="không cắt tỉa")
ax.plot(prune_table["độ sâu"], prune_table["nút: cắt, 1→7"], marker="o", label="cắt tỉa, xét 1→7")
ax.plot(prune_table["độ sâu"], prune_table["nút: cắt, giữa trước"], marker="o", label="cắt tỉa, xét từ giữa")
ax.set_yscale("log")
ax.set_xlabel("độ sâu (số nước nhìn trước sau nước của mình)")
ax.set_ylabel("số nút trung bình (thang log)")
ax.set_title("Ablation cắt tỉa Alpha-Beta")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(ex.fig("nb2_ablation_cat_tia.png"), dpi=150)

## 3. Ablation 2: số mô phỏng của MCTS

Cấp 4 với số mô phỏng mỗi nước thay đổi, đấu cấp 2 (cùng một đối thủ cố định).

In [ ]:
mcts_cfg = cfg["ablation_mcts_simulations"]
sim_list = mcts_cfg["simulations"] if not QUICK else [25, 100]
n_games = ex.n(mcts_cfg["games_per_pair"], 6)
log_path = ex.log("nb2_ablation_mcts.jsonl")
opponent_name = mcts_cfg["opponent"]
opponent = ex.agent(opponent_name)

mcts_summaries = []
for k, sims in enumerate(sim_list):
    name = f"cap4_{sims}mp"
    agent = ex.agent("cap4", overrides={"num_simulations": sims})
    s = run_match(agent, opponent, name, opponent_name, n_games, seed=mcts_cfg["seed"] + k, log_path=log_path)
    mcts_summaries.append(s)
    print(format_summary(s))

mcts_table = summaries_table(mcts_summaries)
mcts_table.insert(1, "mô phỏng/nước", sim_list)
mcts_table.to_csv(ex.table("nb2_ablation_mcts.csv"), index=False, encoding="utf-8-sig")
mcts_table

In [ ]:
fig = plot_scores(sim_list, {"cấp 4 đấu cấp 2": mcts_summaries}, "số mô phỏng mỗi nước",
                  "Ablation số mô phỏng của MCTS", ex.fig("nb2_ablation_mcts.png"))
fig.axes[0].set_xscale("log")
fig.savefig(ex.fig("nb2_ablation_mcts.png"), dpi=150)

## 4. Độ nhạy hàm lượng giá của cấp 3

So bốn hàm lượng giá khi chạm độ sâu, cùng độ sâu 3:
- `quan_x_duong_mo`: hàm hiện tại, mỗi quân cộng 1 cho mỗi đường thắng còn mở đi qua nó, trừ phần của đối thủ, chia 276. Không có tham số nào phải chọn.
- `duong_mo`: chỉ đếm số đường còn mở của mỗi bên (không tính số quân trong đường).
- `trong_so_5_2_3`: bản cũ có trọng số chọn tay (5 cho 3 quân, 2 cho 2 quân, 3 cho mỗi quân ở cột giữa).
- `khong_luong_gia`: trả 0 khi chạm độ sâu, đúng như sách chương 5–6.

Mỗi hàm đấu cấp 2, và ba hàm còn lại đấu trực tiếp hàm hiện tại. Mỗi ván mở đầu bằng 2 nước ngẫu nhiên có seed vì cấp 3 gần như tất định.
Các ván ở đây dùng seed riêng (`ablation_eval.seed`), không trùng với giải đấu cuối ở nb6.

In [ ]:
def eval_open_lines(env):
    """Số đường còn mở của mình trừ số đường còn mở của đối thủ (chỉ tính đường đã có quân)."""
    me = 1 if env.turn == "red" else -1
    score = 0
    for cells in alphabeta.WINDOWS:
        mine = 0
        theirs = 0
        for col, row in cells:
            if env.state[col][row] == me:
                mine += 1
            elif env.state[col][row] == -me:
                theirs += 1
        if mine > 0 and theirs == 0:
            score += 1
        if theirs > 0 and mine == 0:
            score -= 1
    return score / len(alphabeta.WINDOWS)


def eval_weights_523(env):
    """Bản cũ: 5 cho cửa sổ 3 quân, 2 cho cửa sổ 2 quân, 3 cho mỗi quân ở cột giữa."""
    me = 1 if env.turn == "red" else -1
    score = 0
    for cells in alphabeta.WINDOWS:
        mine = 0
        theirs = 0
        for col, row in cells:
            if env.state[col][row] == me:
                mine += 1
            elif env.state[col][row] == -me:
                theirs += 1
        if mine > 0 and theirs > 0:
            continue
        if mine == 3:
            score += 5
        elif mine == 2:
            score += 2
        elif theirs == 3:
            score -= 5
        elif theirs == 2:
            score -= 2
    for row in range(6):
        if env.state[3][row] == me:
            score += 3
        elif env.state[3][row] == -me:
            score -= 3
    return max(-0.9, min(0.9, score / 100.0))


def eval_zero(env):
    return 0.0


EVALUATORS = {
    "quan_x_duong_mo": alphabeta.evaluate,
    "duong_mo": eval_open_lines,
    "trong_so_5_2_3": eval_weights_523,
    "khong_luong_gia": eval_zero,
}


def cap3_with(eval_fn, depth=3):
    """Cấp 3 dùng hàm lượng giá eval_fn: thay tạm alphabeta.evaluate trong lúc tìm kiếm."""
    def agent(env):
        original = alphabeta.evaluate
        alphabeta.evaluate = eval_fn
        try:
            return alphabeta.MiniMax_conn(env, depth=depth)
        finally:
            alphabeta.evaluate = original
    return (agent, module_info(alphabeta))

In [ ]:
eval_cfg = cfg["ablation_eval"]
n_games = ex.n(eval_cfg["games_per_pair"], 6)
opening = eval_cfg["opening_random_moves"]
log_path = ex.log("nb2_ablation_luong_gia.jsonl")
cap2 = with_random_opening(ex.agent(eval_cfg["opponent"]), opening)
current = with_random_opening(cap3_with(EVALUATORS["quan_x_duong_mo"]), opening)

eval_summaries = []
k = 0
for name in eval_cfg["variants"]:
    agent = with_random_opening(cap3_with(EVALUATORS[name]), opening)
    s = run_match(agent, cap2, name, "cap2", n_games, seed=eval_cfg["seed"] + k, log_path=log_path)
    eval_summaries.append(s)
    print(format_summary(s))
    k += 1
    if name != "quan_x_duong_mo":
        s = run_match(agent, current, name, "quan_x_duong_mo", n_games, seed=eval_cfg["seed"] + k, log_path=log_path)
        eval_summaries.append(s)
        print(format_summary(s))
        k += 1

eval_table = summaries_table(eval_summaries)
eval_table.to_csv(ex.table("nb2_ablation_luong_gia.csv"), index=False, encoding="utf-8-sig")
eval_table

## 5. RQ1: cấp 3 và cấp 4 cùng thời gian mỗi nước

Cả hai tác tử được cùng một ngân sách thời gian mỗi nước. Cấp 3 đào sâu dần (độ sâu 0, 1, 2, ...) đến khi hết giờ; cấp 4 mô phỏng đến khi hết giờ.
Ngoài score còn ghi độ sâu trung bình cấp 3 đạt được và số mô phỏng trung bình của cấp 4.

In [ ]:
rq1_cfg = cfg["rq1"]
time_limits = rq1_cfg["time_limits"] if not QUICK else [0.05, 0.1]
n_games = ex.n(rq1_cfg["games_per_pair"], 4)
log_path = ex.log("nb2_rq1.jsonl")

rq1_summaries = []
extra = []
for k, t in enumerate(time_limits):
    name3 = f"cap3_{t}s"
    name4 = f"cap4_{t}s"
    a = ex.agent("cap3", overrides={"time_limit": t})
    b = ex.agent("cap4", overrides={"time_limit": t})
    start = time.perf_counter()
    s = run_match(a, b, name3, name4, n_games, seed=rq1_cfg["seed"] + k, log_path=log_path)
    rq1_summaries.append(s)
    records = records_of(log_path, name3, name4)
    extra.append({
        "giây/nước": t,
        "độ sâu TB cấp 3": info_mean(records, name3, "depth"),
        "số nút TB cấp 3": info_mean(records, name3, "nodes"),
        "số mô phỏng TB cấp 4": info_mean(records, name4, "simulations"),
        "p95 cấp 4 (ms)": round(s["opponent_time_p95"] * 1000, 1),
    })
    print(format_summary(s), "| chạy mất", round(time.perf_counter() - start), "giây")

rq1_table = summaries_table(rq1_summaries)
rq1_table = pd.concat([rq1_table, pd.DataFrame(extra)], axis=1)
rq1_table.to_csv(ex.table("nb2_rq1.csv"), index=False, encoding="utf-8-sig")
rq1_table

In [ ]:
plot_scores(time_limits, {"cấp 3 đấu cấp 4": rq1_summaries}, "thời gian mỗi nước (giây)",
            "RQ1: score của cấp 3 khi đấu cấp 4 cùng thời gian", ex.fig("nb2_rq1.png"),
            ylabel="score của cấp 3")

In [ ]:
# Ảnh hưởng bên đi trước trong RQ1: score của cấp 3 khi cầm đỏ và khi cầm vàng
rows = []
for s in rq1_summaries:
    rows.append({"cặp": f"{s['agent']} vs {s['opponent']}",
                 "score cấp 3 cầm đỏ": round(s["score_as_red"], 3),
                 "score cấp 3 cầm vàng": round(s["score_as_yellow"], 3)})
pd.DataFrame(rows)